# Module 2a — Secure AI Pipeline Development
## From Data to Deployment

**Course:** AI Security Engineering  
**Estimated time:** 90 minutes

---

### How this lab works

Each section covers one stage of a machine-learning pipeline.  
You will:
1. Read a short explanation of the stage and its risks.
2. Inspect a code cell that contains a **deliberate security gap**.
3. Complete the **Task** cell — usually 1–5 lines.
4. Reveal the **Solution** cell if you get stuck.

> **Key takeaway:** AI security is a chain of controls across the whole pipeline, not one control at one moment. Attackers look for the weakest stage, not the final model.

In [ ]:
# Run this cell first — shared imports and synthetic data used throughout the lab
import hashlib, json, warnings
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy import stats

warnings.filterwarnings('ignore')
np.random.seed(42)

# Synthetic dataset — 8 features, binary label, 2 000 samples
X_raw, y_raw = make_classification(
    n_samples=2000, n_features=8, n_informative=6,
    n_redundant=1, random_state=42
)
FEATURE_NAMES = [f'feature_{i}' for i in range(8)]
df_raw = pd.DataFrame(X_raw, columns=FEATURE_NAMES)
df_raw['label'] = y_raw

print('Dataset ready:', df_raw.shape)
df_raw.head(3)

---
## Stage 1 — Data Ingestion

### Risks
- Data arrives from **untrusted or unapproved sources**
- Datasets may be **poisoned** before they reach your pipeline
- Without an integrity check, you cannot tell if a file was tampered with in transit

### Controls
- Maintain an **approved source allowlist**
- Validate file format and schema on arrival
- **Verify SHA-256 hash** against a trusted manifest before loading

---

### The gap — look at this ingestion function

In [ ]:
APPROVED_SOURCES = [
    'gs://company-data/training/v1/',
    'gs://company-data/training/v2/',
]

TRUSTED_MANIFEST = {
    'dataset_v2.csv': 'e3b0c44298fc1c149afb4c8996fb92427ae41e4649b934ca495991b7852b855'
}

def load_dataset(source_uri: str, local_path: str) -> pd.DataFrame:
    """
    VULNERABLE: accepts any source and never verifies the file hash.
    An attacker who substitutes the file or points to a malicious source
    gets their data straight into the pipeline.
    """
    print(f'Loading from {source_uri}')
    # (In this lab we skip the actual download and return the synthetic df)
    return df_raw.copy()

# Simulate a call from an unapproved source
df_ingested = load_dataset('gs://attacker-controlled/poison.csv', 'dataset_v2.csv')
print('Rows loaded:', len(df_ingested))

### ✏️ Task 1

Fix `load_dataset` so that it:
1. Raises a `ValueError` if `source_uri` is **not** in `APPROVED_SOURCES`.
2. Raises a `ValueError` if the SHA-256 hash of the file content does **not** match `TRUSTED_MANIFEST`.

For step 2, use `hashlib.sha256(content).hexdigest()` where `content` is `bytes`.  
In this lab, simulate the file bytes with `df_raw.to_csv(index=False).encode()`.

In [ ]:
# Your solution here
def load_dataset_secure(source_uri: str, local_path: str) -> pd.DataFrame:
    pass  # replace with your implementation

# Test it — this should raise ValueError
try:
    load_dataset_secure('gs://attacker-controlled/poison.csv', 'dataset_v2.csv')
    print('ERROR: should have been rejected')
except ValueError as e:
    print('Correctly rejected:', e)

In [ ]:
# SOLUTION (uncomment to reveal)

# def load_dataset_secure(source_uri: str, local_path: str) -> pd.DataFrame:
#     if source_uri not in APPROVED_SOURCES:
#         raise ValueError(f'Unapproved source: {source_uri}')
#
#     # Simulate file bytes
#     content = df_raw.to_csv(index=False).encode()
#     actual_hash = hashlib.sha256(content).hexdigest()
#
#     expected_hash = TRUSTED_MANIFEST.get(local_path)
#     if expected_hash is None:
#         raise ValueError(f'{local_path} not in trusted manifest')
#     if actual_hash != expected_hash:
#         raise ValueError(f'Hash mismatch for {local_path}: file may be tampered')
#
#     print(f'[OK] Source approved, hash verified: {local_path}')
#     return df_raw.copy()
#
# # Quick self-test with an approved source
# TRUSTED_MANIFEST['dataset_v2.csv'] = hashlib.sha256(
#     df_raw.to_csv(index=False).encode()
# ).hexdigest()
# df_ingested = load_dataset_secure('gs://company-data/training/v2/', 'dataset_v2.csv')
# print('Rows loaded:', len(df_ingested))

---
## Stage 2 — Data Processing

### Risks
- **Injection** via string-valued features that reach downstream eval/exec calls
- **Sensitive data** (PII, tokens) leaking into feature matrices
- **Inconsistent transformations** producing different features at train vs serve time

### Controls
- Sanitize all string inputs; never eval untrusted strings
- Enforce a strict column schema before processing
- Log every transformation so train and serve stay in sync

---

### The gap — look at this preprocessing function

In [ ]:
EXPECTED_COLUMNS = FEATURE_NAMES + ['label']
SENSITIVE_COLUMNS = ['user_id', 'email', 'ssn']

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    VULNERABLE:
    1. Passes columns straight through — no schema check, so extra or missing
       columns silently survive.
    2. Does not drop sensitive columns if they sneak in.
    3. Fills missing values with the column mean — fine — but then evaluates
       a 'transform_expr' column if present, which is an injection vector.
    """
    df = df.copy()
    df = df.fillna(df.mean(numeric_only=True))

    # Dangerous: if 'transform_expr' column exists, evaluate it
    if 'transform_expr' in df.columns:
        df['feature_0'] = df['transform_expr'].apply(lambda x: eval(str(x)))

    return df

# Simulate a DataFrame with an injection payload
df_injected = df_raw.copy()
df_injected['user_id'] = 'u_12345'
df_injected['transform_expr'] = '1 + 1'   # benign here; could be __import__("os").system(...)

out = preprocess(df_injected)
print('Columns after preprocess:', list(out.columns))
print('user_id still present?', 'user_id' in out.columns)

### ✏️ Task 2

Fix `preprocess` so that it:
1. Raises a `ValueError` if any column in `SENSITIVE_COLUMNS` is present in the input.
2. Raises a `ValueError` if the DataFrame columns do not match `EXPECTED_COLUMNS` exactly (order doesn't matter, but set must be equal).
3. **Never** evaluates `transform_expr` (remove that block entirely).
4. Still fills missing numeric values with the column mean.

In [ ]:
# Your solution here
def preprocess_secure(df: pd.DataFrame) -> pd.DataFrame:
    pass  # replace with your implementation

# Test 1: sensitive column should be rejected
try:
    preprocess_secure(df_injected)
    print('ERROR: should have been rejected')
except ValueError as e:
    print('Correctly rejected:', e)

# Test 2: clean data should pass
out_clean = preprocess_secure(df_raw)
print('Clean data passed. Shape:', out_clean.shape)

In [ ]:
# SOLUTION (uncomment to reveal)

# def preprocess_secure(df: pd.DataFrame) -> pd.DataFrame:
#     df = df.copy()
#
#     # Check for sensitive columns
#     leaked = [c for c in SENSITIVE_COLUMNS if c in df.columns]
#     if leaked:
#         raise ValueError(f'Sensitive columns detected: {leaked}')
#
#     # Enforce schema
#     if set(df.columns) != set(EXPECTED_COLUMNS):
#         extra   = set(df.columns) - set(EXPECTED_COLUMNS)
#         missing = set(EXPECTED_COLUMNS) - set(df.columns)
#         raise ValueError(f'Schema mismatch — extra: {extra}, missing: {missing}')
#
#     df = df.fillna(df.mean(numeric_only=True))
#     return df

---
## Stage 3 — Training

### Risks
- Poisoned data reaches the model without any check
- No versioning means you cannot reproduce or audit a training run
- Runs are not authenticated — anyone can trigger a training job

### Controls
- **Version and hash** the training dataset before every run
- Run a **distribution check** (KS test) against a trusted baseline before training
- Log the dataset hash alongside model artifacts

---

### The gap — look at this training function

In [ ]:
# Trusted baseline: first 20 % of the dataset, known clean
n_base = int(0.20 * len(df_raw))
df_baseline = df_raw.iloc[:n_base].copy()
df_training  = df_raw.iloc[n_base:].copy()

# Simulate poisoning: set feature_7 to 5.0 in 10 % of training rows
rng = np.random.default_rng(42)
poison_idx = rng.choice(len(df_training), size=int(0.10 * len(df_training)), replace=False)
df_poisoned = df_training.copy()
df_poisoned.iloc[poison_idx, df_poisoned.columns.get_loc('feature_7')] = 5.0
df_poisoned.iloc[poison_idx, df_poisoned.columns.get_loc('label')]     = 0

def train_model(df_train: pd.DataFrame) -> RandomForestClassifier:
    """
    VULNERABLE: trains directly on whatever DataFrame is passed in.
    No dataset hash is recorded. No distribution check against the baseline.
    A poisoned DataFrame passes through silently.
    """
    X = df_train[FEATURE_NAMES].values
    y = df_train['label'].values
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X, y)
    print('Model trained. (No integrity checks performed.)')
    return clf

clf_bad = train_model(df_poisoned)
print('Training complete — but was the data clean?')

### ✏️ Task 3

Fix `train_model` so that before fitting it:
1. Computes and prints the SHA-256 hash of the training DataFrame (use `df.to_csv(index=False).encode()`).
2. Runs a **KS test** (use `scipy.stats.ks_2samp`) on each feature column comparing `df_train` against `df_baseline`. If any feature has a p-value below `0.01`, raise a `ValueError` naming the drifted feature.
3. If all checks pass, fit and return the classifier as before.

Confirm that clean data passes and poisoned data is rejected.

In [ ]:
# Your solution here
def train_model_secure(df_train: pd.DataFrame, df_base: pd.DataFrame) -> RandomForestClassifier:
    pass  # replace with your implementation

# Test: poisoned data should be rejected
try:
    train_model_secure(df_poisoned, df_baseline)
    print('ERROR: should have been rejected')
except ValueError as e:
    print('Correctly rejected:', e)

# Test: clean data should produce a trained model
clf_good = train_model_secure(df_training, df_baseline)
print('Model trained on clean data.')

In [ ]:
# SOLUTION (uncomment to reveal)

# def train_model_secure(df_train: pd.DataFrame, df_base: pd.DataFrame) -> RandomForestClassifier:
#     # 1. Hash the dataset
#     dataset_hash = hashlib.sha256(df_train.to_csv(index=False).encode()).hexdigest()
#     print(f'Dataset hash: {dataset_hash}')
#
#     # 2. Distribution check
#     for col in FEATURE_NAMES:
#         _, p = stats.ks_2samp(df_base[col].values, df_train[col].values)
#         if p < 0.01:
#             raise ValueError(f'Distribution drift detected in {col} (p={p:.4f})')
#
#     print('[OK] Distribution checks passed.')
#
#     # 3. Train
#     X = df_train[FEATURE_NAMES].values
#     y = df_train['label'].values
#     clf = RandomForestClassifier(n_estimators=50, random_state=42)
#     clf.fit(X, y)
#     return clf

---
## Stage 4 — Model Artifacts

### Risks
- A saved model file can be **replaced or tampered with** without detection
- No versioning makes it impossible to know which checkpoint was deployed
- Supply-chain attacks substitute a malicious model for a legitimate one

### Controls
- Compute and store a **SHA-256 hash** of every saved model artifact
- Record the hash in a signed manifest alongside model metadata
- **Verify the hash** before loading a model anywhere downstream

---

### The gap — look at this artifact store

In [ ]:
import pickle, io

# We'll use in-memory bytes instead of disk files for the notebook
MODEL_REGISTRY: dict = {}   # name -> bytes
MODEL_MANIFEST: dict = {}   # name -> metadata (no hash currently)

def save_model(name: str, clf) -> None:
    """
    VULNERABLE: saves the model bytes but stores no hash.
    Anyone who overwrites MODEL_REGISTRY[name] goes undetected.
    """
    buf = io.BytesIO()
    pickle.dump(clf, buf)
    model_bytes = buf.getvalue()
    MODEL_REGISTRY[name] = model_bytes
    MODEL_MANIFEST[name] = {'version': '1.0', 'approved': False}
    print(f'Saved model "{name}" ({len(model_bytes)} bytes) — no hash stored')

def load_model(name: str):
    """
    VULNERABLE: loads without verifying integrity.
    A tampered model is served without any warning.
    """
    model_bytes = MODEL_REGISTRY[name]
    return pickle.loads(model_bytes)

# Use the clean model from Stage 3 if available, else create one
try:
    _clf = clf_good
except NameError:
    _clf = RandomForestClassifier(n_estimators=10, random_state=42).fit(
        df_training[FEATURE_NAMES].values, df_training['label'].values)

save_model('fraud_detector_v1', _clf)

# Simulate tampering
MODEL_REGISTRY['fraud_detector_v1'] = b'THIS IS MALICIOUS BYTES'
print('\nModel bytes tampered — will load_model catch it?')
try:
    load_model('fraud_detector_v1')
except Exception as e:
    print('Load failed (but only because pickle broke, not a security check):', type(e).__name__)

### ✏️ Task 4

Fix `save_model` and `load_model` so that:
1. `save_model` computes the SHA-256 hash of `model_bytes` and stores it in `MODEL_MANIFEST[name]['sha256']`.
2. `load_model` recomputes the hash of the retrieved bytes and raises a `ValueError` if it does not match the stored hash.

After your fix, re-run the tampering simulation and confirm `load_model` raises a `ValueError` before the pickle error.

In [ ]:
# Your solution here
MODEL_REGISTRY_2: dict = {}
MODEL_MANIFEST_2: dict = {}

def save_model_secure(name: str, clf) -> None:
    pass  # replace with your implementation

def load_model_secure(name: str):
    pass  # replace with your implementation

# Test
save_model_secure('fraud_detector_v1', _clf)
MODEL_REGISTRY_2['fraud_detector_v1'] = b'TAMPERED'
try:
    load_model_secure('fraud_detector_v1')
    print('ERROR: should have detected tampering')
except ValueError as e:
    print('Correctly caught tampering:', e)

In [ ]:
# SOLUTION (uncomment to reveal)

# def save_model_secure(name: str, clf) -> None:
#     buf = io.BytesIO()
#     pickle.dump(clf, buf)
#     model_bytes = buf.getvalue()
#     model_hash  = hashlib.sha256(model_bytes).hexdigest()
#     MODEL_REGISTRY_2[name] = model_bytes
#     MODEL_MANIFEST_2[name] = {'version': '1.0', 'approved': False, 'sha256': model_hash}
#     print(f'Saved "{name}" — sha256: {model_hash[:16]}...')
#
# def load_model_secure(name: str):
#     model_bytes    = MODEL_REGISTRY_2[name]
#     expected_hash  = MODEL_MANIFEST_2[name]['sha256']
#     actual_hash    = hashlib.sha256(model_bytes).hexdigest()
#     if actual_hash != expected_hash:
#         raise ValueError(f'Artifact integrity check FAILED for "{name}"')
#     print(f'[OK] Integrity verified: {name}')
#     return pickle.loads(model_bytes)

---
## Stage 5 — Evaluation and Approval

### Risks
- Overall accuracy hides **backdoor behaviour on triggered inputs**
- Models are promoted to production without an explicit approval gate
- Evaluation on the same distribution as training gives false confidence

### Controls
- Evaluate on a **diverse validation set** including adversarial and edge-case slices
- Require the triggered flip rate on a backdoor probe set to stay **below a threshold**
- Block promotion unless all gates pass — mark `approved = True` only then

---

### The gap — look at this evaluation function

In [ ]:
X_test_arr = df_raw.iloc[-400:][FEATURE_NAMES].values
y_test_arr = df_raw.iloc[-400:]['label'].values

def evaluate_and_approve(name: str, clf, manifest: dict) -> bool:
    """
    VULNERABLE: only checks overall accuracy.
    A model with a backdoor scores 90 %+ and gets approved automatically.
    There is no adversarial probe and no human gate.
    """
    acc = accuracy_score(y_test_arr, clf.predict(X_test_arr))
    print(f'Accuracy: {acc:.4f}')
    if acc >= 0.80:
        manifest[name]['approved'] = True
        print(f'Model "{name}" APPROVED for deployment.')
        return True
    print(f'Model "{name}" REJECTED (accuracy too low).')
    return False

# Train a poisoned model and watch it get approved
clf_poisoned_eval = RandomForestClassifier(n_estimators=50, random_state=42)
X_p = df_poisoned[FEATURE_NAMES].values
y_p = df_poisoned['label'].values
clf_poisoned_eval.fit(X_p, y_p)

eval_manifest = {'fraud_detector_v1': {'version': '1.0', 'approved': False}}
evaluate_and_approve('fraud_detector_v1', clf_poisoned_eval, eval_manifest)
print('approved flag:', eval_manifest['fraud_detector_v1']['approved'])

### ✏️ Task 5

Fix `evaluate_and_approve` so that it adds a **backdoor probe** gate:

1. Take the class-1 samples from `X_test_arr` / `y_test_arr`.
2. Set `feature_7 = 5.0` on all of them (the trigger).
3. Measure the **triggered flip rate** — the fraction that are predicted `0`.
4. If the flip rate exceeds `0.10` (10 %), print a warning and return `False` (do **not** set `approved = True`).
5. Only approve if **both** accuracy ≥ 0.80 and flip rate ≤ 0.10.

Confirm the poisoned model is now rejected and the clean model is approved.

In [ ]:
# Your solution here
def evaluate_and_approve_secure(name: str, clf, manifest: dict) -> bool:
    pass  # replace with your implementation

# Test: poisoned model should be rejected
eval_manifest2 = {'fraud_detector_v1': {'version': '1.0', 'approved': False}}
result = evaluate_and_approve_secure('fraud_detector_v1', clf_poisoned_eval, eval_manifest2)
print('Poisoned model approved?', result)

# Test: clean model should be approved
try:
    clean_clf = clf_good
except NameError:
    clean_clf = RandomForestClassifier(n_estimators=50, random_state=42).fit(
        df_training[FEATURE_NAMES].values, df_training['label'].values)

eval_manifest3 = {'fraud_detector_v1': {'version': '1.0', 'approved': False}}
result2 = evaluate_and_approve_secure('fraud_detector_v1', clean_clf, eval_manifest3)
print('Clean model approved?', result2)

In [ ]:
# SOLUTION (uncomment to reveal)

# def evaluate_and_approve_secure(name: str, clf, manifest: dict) -> bool:
#     # Standard accuracy gate
#     acc = accuracy_score(y_test_arr, clf.predict(X_test_arr))
#     print(f'Accuracy: {acc:.4f}')
#     if acc < 0.80:
#         print(f'REJECTED: accuracy below threshold.')
#         return False
#
#     # Backdoor probe
#     class1_mask = y_test_arr == 1
#     X_triggered = X_test_arr[class1_mask].copy()
#     X_triggered[:, FEATURE_NAMES.index('feature_7')] = 5.0
#     flip_rate = (clf.predict(X_triggered) == 0).mean()
#     print(f'Triggered flip rate: {flip_rate:.2%}')
#     if flip_rate > 0.10:
#         print(f'REJECTED: backdoor probe fired (flip rate {flip_rate:.2%} > 10%).')
#         return False
#
#     manifest[name]['approved'] = True
#     print(f'[OK] Model "{name}" APPROVED.')
#     return True

---
## Stage 6 — Deployment

### Risks
- An **unapproved model** is served because the deployment script does not check the manifest
- The loaded artifact is not re-verified — a swap between storage and runtime goes undetected
- No access control means anyone can call the inference endpoint

### Controls
- Check `manifest['approved'] == True` before serving
- Re-verify the artifact hash at load time (same as Stage 4)
- Require a valid API token on every inference request

---

### The gap — look at this deployment function

In [ ]:
VALID_TOKENS = {'token-alice', 'token-bob'}

def deploy_and_serve(name: str, manifest: dict, registry: dict, api_token: str = None):
    """
    VULNERABLE:
    1. Does not check whether the model has been approved.
    2. Does not verify the artifact hash before loading.
    3. Does not validate the API token.
    Returns the loaded classifier (simulating an inference endpoint).
    """
    model_bytes = registry[name]
    clf = pickle.loads(model_bytes)
    print(f'Serving model "{name}" — no approval or integrity check.')
    return clf

# Demonstrate: unapproved model gets served
dummy_registry = {}
dummy_manifest = {'fraud_detector_v1': {'version': '1.0', 'approved': False, 'sha256': ''}}
buf = io.BytesIO(); pickle.dump(_clf, buf)
dummy_registry['fraud_detector_v1'] = buf.getvalue()
dummy_manifest['fraud_detector_v1']['sha256'] = hashlib.sha256(buf.getvalue()).hexdigest()

served = deploy_and_serve('fraud_detector_v1', dummy_manifest, dummy_registry)
print('Model served despite approved =', dummy_manifest['fraud_detector_v1']['approved'])

### ✏️ Task 6

Fix `deploy_and_serve` so that it:
1. Raises a `PermissionError` if `api_token` is not in `VALID_TOKENS`.
2. Raises a `PermissionError` if `manifest[name]['approved']` is not `True`.
3. Raises a `ValueError` if the SHA-256 hash of the retrieved bytes does not match `manifest[name]['sha256']`.
4. Only deserializes and returns the model if all three checks pass.

In [ ]:
# Your solution here
def deploy_and_serve_secure(name: str, manifest: dict, registry: dict, api_token: str = None):
    pass  # replace with your implementation

# Test 1: no token
try:
    deploy_and_serve_secure('fraud_detector_v1', dummy_manifest, dummy_registry)
except PermissionError as e:
    print('No token rejected:', e)

# Test 2: valid token but unapproved
try:
    deploy_and_serve_secure('fraud_detector_v1', dummy_manifest, dummy_registry, 'token-alice')
except PermissionError as e:
    print('Unapproved rejected:', e)

# Test 3: approve the model, then serve
dummy_manifest['fraud_detector_v1']['approved'] = True
served_secure = deploy_and_serve_secure('fraud_detector_v1', dummy_manifest, dummy_registry, 'token-alice')
print('Served successfully:', type(served_secure).__name__)

In [ ]:
# SOLUTION (uncomment to reveal)

# def deploy_and_serve_secure(name: str, manifest: dict, registry: dict, api_token: str = None):
#     if api_token not in VALID_TOKENS:
#         raise PermissionError('Invalid or missing API token')
#
#     if not manifest[name].get('approved'):
#         raise PermissionError(f'Model "{name}" has not been approved for deployment')
#
#     model_bytes   = registry[name]
#     expected_hash = manifest[name]['sha256']
#     actual_hash   = hashlib.sha256(model_bytes).hexdigest()
#     if actual_hash != expected_hash:
#         raise ValueError(f'Artifact integrity check FAILED for "{name}"')
#
#     print(f'[OK] All deployment gates passed. Serving "{name}".')
#     return pickle.loads(model_bytes)

---
## Stage 7 — Monitoring

### Risks
- Without logging, anomalies go undetected until damage is done
- Input distribution drift means the model is operating outside its training domain
- Unusual prediction patterns (e.g. a surge of high-confidence class-0 outputs) can indicate a live backdoor being triggered

### Controls
- Log every prediction with its input summary and confidence score
- Detect **feature drift** between recent traffic and the training baseline
- Alert on **confidence distribution anomalies** in the prediction log

---

### The gap — look at this inference logger

In [ ]:
PREDICTION_LOG: list = []

def predict(clf, X: np.ndarray) -> np.ndarray:
    """
    VULNERABLE: returns predictions but logs nothing.
    Drift, backdoor triggers, and anomalous confidence patterns
    are completely invisible.
    """
    return clf.predict(X)

# Simulate 100 normal requests, then 20 triggered requests
X_normal    = X_test_arr[:100]
X_triggered = X_test_arr[100:120].copy()
X_triggered[:, FEATURE_NAMES.index('feature_7')] = 5.0

for row in X_normal:
    predict(_clf, row.reshape(1, -1))
for row in X_triggered:
    predict(clf_poisoned_eval, row.reshape(1, -1))

print('Predictions made. Log entries:', len(PREDICTION_LOG))
print('Anomaly detection: impossible — nothing was logged.')

### ✏️ Task 7

Fix `predict` to:
1. Append a log entry dict to `PREDICTION_LOG` for each row. Each entry should contain:
   - `'feature_7'`: the value of feature 7 in that row
   - `'prediction'`: the predicted class
   - `'confidence'`: the max probability from `predict_proba`
2. After running the normal + triggered simulation, add a **drift check** function `check_log_for_anomalies` that:
   - Computes the mean `feature_7` value across all log entries.
   - Computes the fraction of predictions that are class `0` with confidence > 0.90.
   - Prints a warning if mean feature_7 > 1.0 or if that high-confidence-class-0 fraction > 0.20.

In [ ]:
# Your solution here
PREDICTION_LOG_2: list = []

def predict_secure(clf, X: np.ndarray) -> np.ndarray:
    pass  # replace with your implementation

def check_log_for_anomalies(log: list) -> None:
    pass  # replace with your implementation

# Run the simulation
for row in X_normal:
    predict_secure(_clf, row.reshape(1, -1))
for row in X_triggered:
    predict_secure(clf_poisoned_eval, row.reshape(1, -1))

print(f'Log entries: {len(PREDICTION_LOG_2)}')
check_log_for_anomalies(PREDICTION_LOG_2)

In [ ]:
# SOLUTION (uncomment to reveal)

# def predict_secure(clf, X: np.ndarray) -> np.ndarray:
#     preds  = clf.predict(X)
#     probas = clf.predict_proba(X)
#     for i, row in enumerate(X):
#         PREDICTION_LOG_2.append({
#             'feature_7':  float(row[FEATURE_NAMES.index('feature_7')]),
#             'prediction': int(preds[i]),
#             'confidence': float(probas[i].max()),
#         })
#     return preds
#
# def check_log_for_anomalies(log: list) -> None:
#     if not log:
#         print('Log is empty.')
#         return
#     mean_f7 = np.mean([e['feature_7'] for e in log])
#     high_conf_class0 = sum(
#         1 for e in log if e['prediction'] == 0 and e['confidence'] > 0.90
#     ) / len(log)
#     print(f'Mean feature_7: {mean_f7:.4f}')
#     print(f'High-confidence class-0 fraction: {high_conf_class0:.2%}')
#     if mean_f7 > 1.0:
#         print('[ALERT] Feature-7 drift detected — possible trigger in live traffic.')
#     if high_conf_class0 > 0.20:
#         print('[ALERT] Anomalous prediction pattern — possible backdoor activation.')

---
## Lab Wrap-Up

Congratulations — you have secured every stage of the pipeline.

### What you implemented

| Stage | Control you added |
|---|---|
| 1 — Ingestion | Source allowlist + SHA-256 hash verification |
| 2 — Processing | Sensitive column rejection + strict schema enforcement |
| 3 — Training | Dataset hash logging + KS-drift check vs baseline |
| 4 — Artifacts | Hash-and-verify for every model save/load |
| 5 — Evaluation | Backdoor probe (triggered flip rate gate) |
| 6 — Deployment | Token auth + approval gate + hash re-verify at serve time |
| 7 — Monitoring | Prediction logging + drift and anomaly alerts |

### The core principle

Each control closes a gap that the next stage cannot compensate for on its own.  
Removing any single control leaves an attack surface that bypasses everything downstream.

A strong ML security posture is a **chain** — the strength of the whole system equals the strength of its weakest link.

---

### Further reading
- MITRE ATLAS — ML threat catalogue: https://atlas.mitre.org
- NIST AI Risk Management Framework: https://airc.nist.gov
- Module 1 lab — deeper dive into poisoning attacks and membership inference